<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/xGBoostModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score,classification_report
from xgboost import XGBClassifier
import pickle
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score
from sklearn.metrics import confusion_matrix

In [3]:
!pip install --upgrade pandas numpy

In [5]:
import os

print(os.path.getsize("sparse_dict.pkl"))

786883810


In [1]:
#Load the sparse dataset
import pickle
with open("sparse_dict.pkl", "rb") as f:
  sparse = pickle.load(f)


ValueError: unregistered extension code 11

In [3]:
print(sparse.keys())

dict_keys(['sparse_train', 'sparse_val', 'sparse_test'])


In [4]:
for name, hts in sparse.items():
  print(name, hts.shape)

sparse_train (1271440, 56)
sparse_val (423814, 56)
sparse_test (423814, 56)


In [19]:
cols_to_drop = ["EMR", "SDP", "County","PatientPk","MFL_Code","SiteCode","Latitude","Longitude","Facility_Name", "SubCounty",
                "ClientTestedAs","CoupleDiscordant","VisitDate","NumberOfPartners"]

for key in sparse.keys():
  sparse[key] = sparse[key].drop(columns=cols_to_drop, errors="ignore")


In [28]:
#Check dtypes
for name, df in sparse.items():
    print(f"\n{name}")
    print(df.dtypes.value_counts())



sparse_train
object     41
float64     1
Name: count, dtype: int64

sparse_val
object     41
float64     1
Name: count, dtype: int64

sparse_test
object     41
float64     1
Name: count, dtype: int64


Prepare train, validation and test sets

In [20]:
#define features and target variable
X_train = sparse["sparse_train"].drop(columns=["FinalTestResult"])
y_train = sparse["sparse_train"]["FinalTestResult"]

X_val = sparse["sparse_val"].drop(columns=["FinalTestResult"])
y_val = sparse["sparse_val"]["FinalTestResult"]

X_test = sparse["sparse_test"].drop(columns=["FinalTestResult"])
y_test = sparse["sparse_test"]["FinalTestResult"]

Encode categorical variables

In [21]:
dtype_summary = pd.DataFrame({
    "Variable": X_train.columns,
    "Datatype": X_train.dtypes.astype(str)
})

dtype_summary = dtype_summary.sort_values("Datatype")

dtype_summary.head(100)

,Variable,Datatype
Age,Age,float64
PopulationType,PopulationType,object
ResultOfHIVSelf,ResultOfHIVSelf,object
ScreenedTB,ScreenedTB,object
TBStatus,TBStatus,object
ReceivedServices,ReceivedServices,object
ViolenceScreeningType,ViolenceScreeningType,object
Sex,Sex,object
MaritalStatus,MaritalStatus,object
PatientDisabled,PatientDisabled,object


In [22]:
for col in X_train.columns:
    print(col, X_train[col].nunique())


PopulationType 3
KeyPopulation 5
PriorityPopulation 1
IsHealthWorker 3
TestedHIVBefore 2
ResultOfHIV 2
EverHadSex 3
SexuallyActive 5
NewPartner 4
PartnerHIVStatus 5
MultiplePartners 3
AlcoholSex 5
MoneySex 4
CondomBurst 4
UnknownStatusPartner 4
KnownStatusPartner 4
Pregnant 4
BreastfeedingMother 4
ExperiencedViolenceScreening 4
CurrentlyOnPrep 3
TraditionalProcedures 2
MothersStatus 4
ResultOfHIVSelf 2
ScreenedTB 3
TBStatus 4
ReceivedServices 9
ViolenceScreeningType 9
Sex 2
MaritalStatus 7
PatientDisabled 2
EverTestedForHiv 2
MonthsSinceLastTest 5
PriorityPopulationType 5
Age 99
GBVSexual 3
GBVPhysical 3
GBVEmotional 3
ReceivedPREP 2
ReceivedTB 2
ReceivedSTI 2
dayofweek 7


In [23]:
#One_Hot encoding
categorical_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns

combined = pd.concat([X_train, X_val, X_test],axis=0)

combined_enc = pd.get_dummies(combined,columns=categorical_cols,dummy_na=True)

n_train = len(X_train)
n_val = len(X_val)

X_train_enc = combined_enc.iloc[:n_train]
X_val_enc = combined_enc.iloc[n_train:n_train+n_val]
X_test_enc = combined_enc.iloc[n_train+n_val:]

In [25]:
#Verify that all columns match
print(X_train_enc.columns.equals(X_val_enc.columns))
print(X_train_enc.columns.equals(X_test_enc.columns))

True
True


In [26]:
X_train_enc.head()

,Age,PopulationType_GP,PopulationType_KP,PopulationType_PRIORITY,PopulationType_nan,KeyPopulation_FSW,KeyPopulation_MSM,KeyPopulation_NR,KeyPopulation_PRISONER,KeyPopulation_PWID,KeyPopulation_nan,PriorityPopulation_NR,PriorityPopulation_nan,IsHealthWorker_NO,IsHealthWorker_NR,IsHealthWorker_YES,IsHealthWorker_nan,TestedHIVBefore_NO,TestedHIVBefore_YES,TestedHIVBefore_nan,ResultOfHIV_NEGATIVE,ResultOfHIV_UNKNOWN,ResultOfHIV_nan,EverHadSex_NO,EverHadSex_NR,EverHadSex_YES,EverHadSex_nan,SexuallyActive_DECLINED,SexuallyActive_NAN,SexuallyActive_NO,SexuallyActive_NR,SexuallyActive_YES,SexuallyActive_nan,NewPartner_DECLINED,NewPartner_NO,NewPartner_NR,NewPartner_YES,NewPartner_nan,PartnerHIVStatus_DECLINED,PartnerHIVStatus_NEGATIVE,...,MonthsSinceLastTest_MORETHANTWOYEARS,MonthsSinceLastTest_NR,MonthsSinceLastTest_ONETOTWOYEARS,MonthsSinceLastTest_SEVENTOTWELVE,MonthsSinceLastTest_nan,PriorityPopulationType_Adolescent and young girls,PriorityPopulationType_Fisher folk,PriorityPopulationType_Not applicable,PriorityPopulationType_Prisoner,PriorityPopulationType_Truck driver,PriorityPopulationType_nan,GBVSexual_NAN,GBVSexual_NO,GBVSexual_YES,GBVSexual_nan,GBVPhysical_NAN,GBVPhysical_NO,GBVPhysical_YES,GBVPhysical_nan,GBVEmotional_NAN,GBVEmotional_NO,GBVEmotional_YES,GBVEmotional_nan,ReceivedPREP_NO,ReceivedPREP_YES,ReceivedPREP_nan,ReceivedTB_NO,ReceivedTB_YES,ReceivedTB_nan,ReceivedSTI_NO,ReceivedSTI_YES,ReceivedSTI_nan,dayofweek_FRIDAY,dayofweek_MONDAY,dayofweek_SATURDAY,dayofweek_SUNDAY,dayofweek_THURSDAY,dayofweek_TUESDAY,dayofweek_WEDNESDAY,dayofweek_nan
1468429,21.0,True,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False
428004,23.0,True,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,True,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,True,True,False,False,False,False,False,False,False
1630362,25.0,False,True,False,False,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False
330368,20.0,False,False,True,False,False,False,True,False,False,False,False,True,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,False,False,...,False,True,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False
973348,37.0,True,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,True,False,False,False,True,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,False,False,False,False


In [27]:
X_train_enc.dtypes.value_counts()


,count
bool,190
float64,1


In [31]:
#convert bools to int
for df in [X_train_enc, X_val_enc, X_test_enc]:

    bool_cols = df.select_dtypes(include='bool').columns

    df[bool_cols] = df[bool_cols].astype('int8')

/tmp/ipykernel_541/2293876107.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[bool_cols] = df[bool_cols].astype('int8')
/tmp/ipykernel_541/2293876107.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[bool_cols] = df[bool_cols].astype('int8')
/tmp/ipykernel_541/2293876107.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do

In [32]:
X_train_enc.dtypes.value_counts()

,count
int8,190
float64,1


In [33]:
X_train_enc.head()

,Age,PopulationType_GP,PopulationType_KP,PopulationType_PRIORITY,PopulationType_nan,KeyPopulation_FSW,KeyPopulation_MSM,KeyPopulation_NR,KeyPopulation_PRISONER,KeyPopulation_PWID,KeyPopulation_nan,PriorityPopulation_NR,PriorityPopulation_nan,IsHealthWorker_NO,IsHealthWorker_NR,IsHealthWorker_YES,IsHealthWorker_nan,TestedHIVBefore_NO,TestedHIVBefore_YES,TestedHIVBefore_nan,ResultOfHIV_NEGATIVE,ResultOfHIV_UNKNOWN,ResultOfHIV_nan,EverHadSex_NO,EverHadSex_NR,EverHadSex_YES,EverHadSex_nan,SexuallyActive_DECLINED,SexuallyActive_NAN,SexuallyActive_NO,SexuallyActive_NR,SexuallyActive_YES,SexuallyActive_nan,NewPartner_DECLINED,NewPartner_NO,NewPartner_NR,NewPartner_YES,NewPartner_nan,PartnerHIVStatus_DECLINED,PartnerHIVStatus_NEGATIVE,...,MonthsSinceLastTest_MORETHANTWOYEARS,MonthsSinceLastTest_NR,MonthsSinceLastTest_ONETOTWOYEARS,MonthsSinceLastTest_SEVENTOTWELVE,MonthsSinceLastTest_nan,PriorityPopulationType_Adolescent and young girls,PriorityPopulationType_Fisher folk,PriorityPopulationType_Not applicable,PriorityPopulationType_Prisoner,PriorityPopulationType_Truck driver,PriorityPopulationType_nan,GBVSexual_NAN,GBVSexual_NO,GBVSexual_YES,GBVSexual_nan,GBVPhysical_NAN,GBVPhysical_NO,GBVPhysical_YES,GBVPhysical_nan,GBVEmotional_NAN,GBVEmotional_NO,GBVEmotional_YES,GBVEmotional_nan,ReceivedPREP_NO,ReceivedPREP_YES,ReceivedPREP_nan,ReceivedTB_NO,ReceivedTB_YES,ReceivedTB_nan,ReceivedSTI_NO,ReceivedSTI_YES,ReceivedSTI_nan,dayofweek_FRIDAY,dayofweek_MONDAY,dayofweek_SATURDAY,dayofweek_SUNDAY,dayofweek_THURSDAY,dayofweek_TUESDAY,dayofweek_WEDNESDAY,dayofweek_nan
1468429,21.0,1,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0
428004,23.0,1,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0
1630362,25.0,0,1,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0
330368,20.0,0,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,...,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0
973348,37.0,1,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0


Encode target variable

In [34]:
#Check the output, if not in numericals, encode it
print(y_train.value_counts())

FinalTestResult
NEGATIVE    1239327
POSITIVE      32113
Name: count, dtype: int64


In [38]:
#Label Ebcode target variable
from sklearn.preprocessing import LabelEncoder
y_train_enc = LabelEncoder().fit_transform(y_train)
y_val_enc = LabelEncoder().fit_transform(y_val)
y_test_enc = LabelEncoder().fit_transform(y_test)

In [39]:
#Calculate this if positivity is much fewer than negatives: This is because HIV positivity is usually imbalanced
neg = (y_train_enc == 0).sum()
pos = (y_train_enc == 1).sum()

scale_pos_weight = neg / pos

print(scale_pos_weight)

38.592688319372215


Train XGBoost

In [40]:
pip install xgboost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=2,4,6,8,
    learning_rate=[0.01,0.05,0.1],
    scale_pos_weight=[1,10,33],
    random_state=2231,
    eval_metric="aucpr",
    early_stopping_rounds=30
)

xgb_model.fit(
    X_train_enc,
    y_train_enc,
    eval_set=[(X_val_enc, y_val_enc)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

Validate the model

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)

val_pred = xgb_model.predict(X_val_enc)
val_prob = xgb_model.predict_proba(X_val_enc)[:,1]

In [ ]:
#Check Accuracy
print("Validation Accuracy:",
      accuracy_score(y_val_enc, val_pred))

Validation Accuracy: 0.7923994960053231


In [ ]:
#Check AUC
print("Validation AUC:",
      roc_auc_score(y_val_enc, val_prob))

Validation AUC: 0.8675413385836388


In [ ]:
print(classification_report(y_val_enc, val_pred))

              precision    recall  f1-score   support

           0       0.99      0.79      0.88    413110
           1       0.09      0.77      0.16     10704

    accuracy                           0.79    423814
   macro avg       0.54      0.78      0.52    423814
weighted avg       0.97      0.79      0.86    423814



The model as is has a recall of 77%. Predicts 77% of people to turn positive but only 9% of the people turns positive leading to so many false postives

In [ ]:
#Check ROC-AUC
from sklearn.metrics import roc_auc_score

val_prob = xgb_model.predict_proba(X_val_enc)[:,1]

roc_auc_score(y_val_enc, val_prob)

np.float64(0.8675413385836388)

In [ ]:
#Check PR-AUC
from sklearn.metrics import average_precision_score

average_precision_score(y_val_enc, val_prob)

np.float64(0.25388613997138554)

In [ ]:
#Confusion Matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val_enc, val_pred)
print(cm)

[[327588  85522]
 [  2462   8242]]


Retrain the model

In [ ]:
#Retrain with validation monitoring

xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    random_state=2231,

    n_estimators=2000,
    learning_rate=0.03,

    max_depth=4,
    min_child_weight=5,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=1,
    reg_lambda=5
)

xgb_model.fit(
    X_train_enc,
    y_train_enc,
    eval_set=[(X_val_enc, y_val_enc)],
    verbose=100
)

[0]	validation_0-aucpr:0.08578
[100]	validation_0-aucpr:0.18374
[200]	validation_0-aucpr:0.20579
[300]	validation_0-aucpr:0.21985
[400]	validation_0-aucpr:0.22917
[500]	validation_0-aucpr:0.23729
[600]	validation_0-aucpr:0.24269
[700]	validation_0-aucpr:0.24788
[800]	validation_0-aucpr:0.25267
[900]	validation_0-aucpr:0.25589
[1000]	validation_0-aucpr:0.25957
[1100]	validation_0-aucpr:0.26287
[1200]	validation_0-aucpr:0.26547
[1300]	validation_0-aucpr:0.26836
[1400]	validation_0-aucpr:0.27098
[1500]	validation_0-aucpr:0.27364
[1600]	validation_0-aucpr:0.27585
[1700]	validation_0-aucpr:0.27807
[1800]	validation_0-aucpr:0.27998
[1900]	validation_0-aucpr:0.28182
[1999]	validation_0-aucpr:0.28361


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=2000,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report
)

val_prob = xgb_model.predict_proba(X_val_enc)[:,1]

print("ROC-AUC:",
      roc_auc_score(y_val_enc, val_prob))

print("PR-AUC:",
      average_precision_score(y_val_enc, val_prob))

ROC-AUC: 0.8721729128269401
PR-AUC: 0.28364486674294365


In [ ]:
val_pred = (val_prob >= 0.5).astype(int)

print(classification_report(y_val_enc, val_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99    413110
           1       0.72      0.07      0.13     10704

    accuracy                           0.98    423814
   macro avg       0.85      0.54      0.56    423814
weighted avg       0.97      0.98      0.97    423814



In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.05,0.10,0.15,0.20,0.25,0.30,0.40,0.50]

for t in thresholds:

    pred = (val_prob >= t).astype(int)

    print(
        f"Threshold={t:.2f}",
        f"Precision={precision_score(y_val_enc,pred):.3f}",
        f"Recall={recall_score(y_val_enc,pred):.3f}",
        f"F1={f1_score(y_val_enc,pred):.3f}"
    )

Threshold=0.05 Precision=0.143 Recall=0.629 F1=0.233
Threshold=0.10 Precision=0.237 Recall=0.447 F1=0.310
Threshold=0.15 Precision=0.316 Recall=0.338 F1=0.327
Threshold=0.20 Precision=0.392 Recall=0.262 F1=0.314
Threshold=0.25 Precision=0.459 Recall=0.208 F1=0.286
Threshold=0.30 Precision=0.522 Recall=0.168 F1=0.254
Threshold=0.40 Precision=0.624 Recall=0.112 F1=0.190
Threshold=0.50 Precision=0.717 Recall=0.073 F1=0.132
